# Install Dependencies

In [ ]:
%pip install javalang gitpython pandas

# Imports

In [ ]:
import os
import glob
import subprocess
import json
import random
import shutil
from pathlib import Path

import pandas as pd
import javalang
from javalang.tokenizer import tokenize

# Configuration


In [ ]:
CLONE_DIR = "/content/java_repos"
OUTPUT_DIR = "/content/dataset"

NUM_REPOS = 700
FILES_PER_REPO = 100

MIN_TOKENS = 10
MAX_TOKENS = 500

TRAIN_TARGET = 50000
VAL_TARGET = 1000

random.seed(42)

os.makedirs(CLONE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete!")
print(f"Clone directory: {CLONE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

# Fetch Repos

In [ ]:
import requests

def fetch_top_java_repos(num_repos=700, per_page=100):
    repos = []
    page = 1

    while len(repos) < num_repos:
        url = "https://api.github.com/search/repositories"
        params = {
            "q": "language:java stars:>500 size:>500",
            "sort": "stars",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }

        response = requests.get(url, params=params)
        if response.status_code != 200:
            break

        data = response.json()
        items = data.get("items", [])

        if not items:
            break

        for item in items:
            if item.get("fork", False):
                continue

            repos.append({
                "full_name": item["full_name"],
                "clone_url": item["clone_url"],
                "stars": item["stargazers_count"],
                "description": item.get("description", "")
            })

        page += 1

        if len(repos) >= num_repos:
            break

    return repos[:num_repos]

# Fetch repositories
print("Fetching top Java repositories from GitHub...")
repo_data = fetch_top_java_repos(num_repos=700)
df_repos = pd.DataFrame(repo_data)

# Split repos by rank (GitHub API order)
TRAIN_REPOS = set(df_repos.iloc[0:600]["full_name"])
VAL_REPOS = set(df_repos.iloc[600:700]["full_name"])

print(f"\nFetched {len(df_repos)} repositories")
print(f"\nTop 10 repos by stars:")
df_repos.head(10)

# Clone Repos

In [ ]:
def clone_repo(clone_url, dest_dir):
    """
    Shallow clone a repository.
    Returns True if successful, False otherwise.
    """
    try:
        if os.path.exists(dest_dir):
            shutil.rmtree(dest_dir)

        cmd = ["git", "clone", "--depth", "1", "--quiet", clone_url, dest_dir]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)

        return result.returncode == 0
    except subprocess.TimeoutExpired:
        print(f"  Timeout cloning {clone_url}")
        return False
    except Exception as e:
        print(f"  Error: {e}")
        return False

# Clone repositories
cloned_repos = []
failed_repos = []

print(f"Cloning {len(df_repos)} repositories...\n")

for idx, row in df_repos.iterrows():
    repo_name = row["full_name"]
    clone_url = row["clone_url"]

    safe_name = repo_name.replace("/", "_")
    dest_dir = os.path.join(CLONE_DIR, safe_name)

    print(f"[{idx+1}/{len(df_repos)}] Cloning {repo_name}...", end=" ")

    success = clone_repo(clone_url, dest_dir)

    if success:
        cloned_repos.append({
            "repo_name": repo_name,
            "local_path": dest_dir,
            "stars": row["stars"]
        })
        print("done")
    else:
        failed_repos.append(repo_name)
        print("failed")

print(f"\n\nSummary:")
print(f"  Successfully cloned: {len(cloned_repos)}")
print(f"  Failed: {len(failed_repos)}")

# Find and Select Java files

In [ ]:
def find_java_files(repo_path):
    """
    Find all .java files in a repository.
    Excludes test files and common non-source directories.
    """
    java_files = []
    exclude_patterns = ["test", "tests", "example", "examples", "sample", "demo", "generated"]

    for root, dirs, files in os.walk(repo_path):
        root_lower = root.lower()
        if any(pattern in root_lower for pattern in exclude_patterns):
            continue

        for file in files:
            if file.endswith(".java"):
                java_files.append(os.path.join(root, file))

    return java_files


def select_java_files(java_files, max_files):
    """
    Randomly select up to max_files from the list.
    """
    if len(java_files) <= max_files:
        return java_files
    return random.sample(java_files, max_files)


# Find and select Java files from each repo
repo_java_files = {}
all_selected_files = []

print(f"Finding Java files (selecting up to {FILES_PER_REPO} per repo)...\n")

for repo_info in cloned_repos:
    repo_name = repo_info["repo_name"]
    repo_path = repo_info["local_path"]

    java_files = find_java_files(repo_path)

    if not java_files:
        print(f"  {repo_name}: No Java files found")
        continue

    selected = select_java_files(java_files, max_files=FILES_PER_REPO)

    repo_java_files[repo_name] = {
        "total_files": len(java_files),
        "selected_files": [os.path.relpath(f, repo_path) for f in selected],
        "remaining_files": len(java_files) - len(selected)
    }

    all_selected_files.extend([(repo_name, f) for f in selected])
    print(f"  {repo_name}: {len(selected)}/{len(java_files)} files selected")

print(f"\nTotal Java files selected: {len(all_selected_files)}")

# Extract Methods and Javadoc

In [ ]:
def read_file_content(file_path):
    """Read file content with multiple encoding fallbacks."""
    encodings = ['utf-8', 'latin-1', 'cp1252']

    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                return f.read()
        except UnicodeDecodeError:
            continue

    return None

import re

def extract_javadoc_summary(lines, start_line):
    """
    Extract the Javadoc summary immediately above a method.
    Handles annotations, blank lines, single-line Javadoc, and multi-line blocks.
    """
    i = start_line - 1

    # Skip annotations and blank lines
    while i >= 0 and (lines[i].strip().startswith("@") or lines[i].strip() == ""):
        i -= 1

    # Single line summaries
    if i >= 0 and lines[i].strip().startswith("/**") and "*/" in lines[i]:
        text = lines[i].strip()[3:-2].strip()
        return text.split(".")[0].strip() if text else None

    # Multi-line summaries
    if i >= 0 and lines[i].strip().endswith("*/"):
        javadoc = []
        j = i
        while j >= 0 and "/**" not in lines[j]:
            javadoc.append(lines[j])
            j -= 1
        if j >= 0:
            javadoc.append(lines[j])
            javadoc.reverse()

            # Clean markers
            text = "\n".join(javadoc)
            text = re.sub(r"/\*\*|\*/", "", text)
            text = re.sub(r"^\s*\*", "", text, flags=re.MULTILINE)
            text = re.sub(r"@\w+.*", "", text)
            text = re.sub(r"\s+", " ", text).strip()

            return text.split(".")[0].strip() if text else None

    return None

def extract_method_source(source_code, method_node, lines):
    """Extract the source code of a method by counting braces."""
    try:
        start_line = method_node.position.line - 1

        brace_count = 0
        started = False
        end_line = start_line

        for i in range(start_line, len(lines)):
            line = lines[i]
            for char in line:
                if char == '{':
                    brace_count += 1
                    started = True
                elif char == '}':
                    brace_count -= 1

            if started and brace_count == 0:
                end_line = i
                break

        method_lines = lines[start_line:end_line + 1]
        return '\n'.join(method_lines)

    except Exception:
        return None

def extract_methods_with_summaries(file_path, repo_name):
    """
    Parse a Java file and extract all methods with their Javadoc summaries.
    """
    methods = []

    source_code = read_file_content(file_path)
    if source_code is None:
        return methods

    lines = source_code.split('\n')

    try:
        tree = javalang.parse.parse(source_code)

        for path, node in tree.filter(javalang.tree.MethodDeclaration):
            if not node.position:
                continue

            method_source = extract_method_source(source_code, node, lines)
            if not method_source:
                continue

            start_line = node.position.line - 1
            summary = extract_javadoc_summary(lines, start_line)

            # Only keep methods that have a Javadoc summary
            if not summary:
                continue

            methods.append({
                "repo": repo_name,
                "file": os.path.basename(file_path),
                "method_name": node.name,
                "source": method_source,
                "summary": summary
            })

    except javalang.parser.JavaSyntaxError:
        pass
    except Exception:
        pass

    return methods

# Extract methods with summaries from all selected files
all_methods = []

print(f"Extracting methods with Javadoc summaries from {len(all_selected_files)} files...\n")

for i, (repo_name, file_path) in enumerate(all_selected_files):
    if (i + 1) % 100 == 0:
        print(f"  Processed {i + 1}/{len(all_selected_files)} files...")

    methods = extract_methods_with_summaries(file_path, repo_name)
    all_methods.extend(methods)

print(f"\nTotal methods with summaries extracted: {len(all_methods)}")


# Filter Methods

In [ ]:
def contains_non_ascii(text):
    """Check if text contains non-ASCII characters."""
    try:
        text.encode('ascii')
        return False
    except UnicodeEncodeError:
        return True


def count_tokens(source_code):
    """Count the number of Java tokens in source code."""
    try:
        tokens = list(tokenize(source_code))
        return len(tokens)
    except:
        return 0

def is_get(method_name):
    return method_name.startswith("get")

def is_set(method_name):
    return method_name.startswith("set")

filtered_methods = []

stats = {
    "total": len(all_methods),
    "non_ascii_dropped": 0,
    "too_short_dropped": 0,
    "too_long_dropped": 0,
    "summary_short_dropped": 0,
    "kept": 0
}

print(f"Filtering {len(all_methods)} methods...\n")

for method in all_methods:
    source = method["source"]
    summary = method["summary"]

    if contains_non_ascii(source) or contains_non_ascii(summary):
        stats["non_ascii_dropped"] += 1
        continue

    token_count = count_tokens(source)
    if token_count < MIN_TOKENS:
        stats["too_short_dropped"] += 1
        continue

    if token_count > MAX_TOKENS:
        stats["too_long_dropped"] += 1
        continue

    # Drop extremely short summaries (e.g., 1–2 words)
    if len(summary.split()) < 3:
        stats["summary_short_dropped"] += 1
        continue

    method["token_count"] = token_count
    filtered_methods.append(method)
    stats["kept"] += 1

print(f"Filtering Results:")
print(f"  Total methods:                 {stats['total']}")
print(f"  Dropped (non-ASCII):           {stats['non_ascii_dropped']}")
print(f"  Dropped (< {MIN_TOKENS} tokens): {stats['too_short_dropped']}")
print(f"  Dropped (> {MAX_TOKENS} tokens): {stats['too_long_dropped']}")
print(f"  Dropped (short summary):       {stats['summary_short_dropped']}")
print(f"  -------------------------")
print(f"  Methods kept:                  {stats['kept']}")

# Flatten Code

In [ ]:
def flatten_code(code):
    return " ".join(code.split())

def normalize_summary(summary):
    return summary.lower().strip()

final_pairs = []
for m in filtered_methods:
    final_pairs.append({
        "code": flatten_code(m["source"]),
        "summary": normalize_summary(m["summary"])
    })


# Generate .txt files for CodeT5

In [ ]:
train_pairs = final_pairs[:TRAIN_TARGET]
val_pairs = final_pairs[TRAIN_TARGET:TRAIN_TARGET + VAL_TARGET]

with open("/content/train_code.txt", "w") as f_code, \
     open("/content/train_summary.txt", "w") as f_sum:
    for p in train_pairs:
        f_code.write(p["code"] + "\n")
        f_sum.write(p["summary"] + "\n")

with open("/content/val_code.txt", "w") as f_code, \
     open("/content/val_summary.txt", "w") as f_sum:
    for p in val_pairs:
        f_code.write(p["code"] + "\n")
        f_sum.write(p["summary"] + "\n")
